# HyperCube ML Workspace Quickstart

이 컨테이너는 시작될 때 다음을 자동으로 했어요:

1. 마운트된 `*.tar.gz` 모델 아카이브를 `/workspace/<asset-slug>/`에 추출
2. `HC_LAUNCHER_RECIPE` 가 설정돼 있으면 백그라운드로 해당 recipe 의 transformers 로드 + gradio UI(:7860) 자동 실행
3. `jupyter-server-proxy` 가 그 gradio 를 현재 워크스페이스 URL 안의 `/proxy/7860/` 로 노출

아래 셀을 한 번 실행하면 gradio UI 가 노트북 출력 영역에 inline iframe 으로 뜹니다. 모델이 로드되는 동안에는 503 / loading 이 잠깐 뜰 수 있어요.

In [ ]:
import os, time, urllib.request, urllib.error
from IPython.display import IFrame, display, Markdown

RECIPE = os.environ.get('HC_LAUNCHER_RECIPE', 'none')
GRADIO_URL = 'proxy/7860/'

if RECIPE == 'none':
    display(Markdown('이 컨테이너에는 자동 추론 레시피가 설정돼 있지 않습니다. 모델 로드는 아래 셀을 참고해 직접 수행하세요.'))
else:
    for attempt in range(60):
        try:
            with urllib.request.urlopen('http://127.0.0.1:7860/', timeout=1) as r:
                if r.status < 500:
                    break
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            pass
        if attempt == 0:
            display(Markdown(f'모델({RECIPE}) 로딩 중 (보통 30~60초)...'))
        time.sleep(2)
    display(Markdown(f'### Gradio UI ({RECIPE})'))
    display(IFrame(src=GRADIO_URL, width='100%', height=720))

## 직접 Python 에서 호출

백그라운드 gradio 가 이미 로딩했더라도, 노트북에서 별도로 transformers 를 호출할 수 있습니다 (메모리 ↑).

In [ ]:
# import os, torch
# from transformers import AutoProcessor, AutoModelForCausalLM
# MODEL = os.environ.get('HC_MODEL_DIR') or '/workspace/<asset-slug>'
# processor = AutoProcessor.from_pretrained(MODEL)
# model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='cuda')